In [1]:
import polars as pl
from uuid import uuid4
from pathlib import Path
import os
import datetime as dt
import json
from typing import List
import logging

from pap_datalab.models.metadata import DepdevPsgcPublications
from pydantic import ValidationError

logging.basicConfig()
_logger = logging.getLogger()
_logger.setLevel(logging.DEBUG)

# 🥉 Bronze
## Parameters

In [2]:
bronze_data_directory = "../data/bronze/depdev/"

files = os.listdir(bronze_data_directory)
metadata_files = [file for file in files if "metadata.json" in file]
data_files = [file for file in files if "metadata.json" not in file]
valid_metadata: List[DepdevPsgcPublications] = []

for metadata_file in metadata_files:
    with open(
        Path(bronze_data_directory) / metadata_file, mode="r", encoding="utf8"
    ) as f:
        contents = f.read()
        maybe_valid_metadata_file = json.loads(contents)
        try:
            valid_datafile = DepdevPsgcPublications.model_validate(
                maybe_valid_metadata_file
            )
            valid_metadata.append(valid_datafile)
        except ValidationError as e:
            _logger.debug(
                f"Skipping associated data-file due to invalid metadata: {metadata_file}"
            )
            print(e)

DEBUG:root:Skipping associated data-file due to invalid metadata: PSGC-2Q-2025-Publication-Datafile.xlsx.metadata.json


1 validation error for DepdevPsgcPublications
column_mappings
  Field required [type=missing, input_value={'name': 'PSGC-3Q-2025-Pu...', 'sheet_name': 'PSGC'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.11/v/missing


# 🥉Bronze -> 🥈Silver

In [3]:
metadata = valid_metadata[0]

In [4]:
DepdevPsgcPublications.model_fields

{'name': FieldInfo(annotation=str, required=True),
 'column_mappings': FieldInfo(annotation=Dict[str, str], required=True),
 'valid_from': FieldInfo(annotation=AwareDatetime, required=True),
 'sheet_name': FieldInfo(annotation=str, required=True)}

In [5]:
# reading the excel file from bronze
df = pl.read_excel(source=bronze_data_directory + metadata.name, sheet_name=metadata.sheet_name)
renamed_df = df.rename(metadata.column_mappings)

## col: `psgc_id`

In [6]:
# verify that all psgc_id have length 10
assert len(renamed_df["psgc_id"].str.len_chars().value_counts()["psgc_id"])==1
assert renamed_df["psgc_id"].str.len_chars().value_counts().row(0)[0] == 10

renamed_df.sample(10)

psgc_id,psgc_name,correspondence_code,geographic_level,old_name,city_class,income_classification,settlement_type,population,remarks,status
str,str,i64,str,str,str,str,str,i64,str,str
"""1004310011""","""Poblacion""",104310011,"""Bgy""",null,null,null,"""U""",7406,null,null
"""0506215002""","""Peñafrancia""",56215002,"""Bgy""",null,null,null,"""R""",842,null,null
"""0907319007""","""Bogo Capalaran""",97319007,"""Bgy""",null,null,null,"""R""",2721,null,null
"""0201529035""","""Centro 2 """,21529035,"""Bgy""",null,null,null,"""U""",592,null,"""Pob."""
"""0603029004""","""AGROCEL Pob.""",63029004,"""Bgy""","""Aguinaldo-Roxas--Celso Mayor""",null,null,"""R""",627,null,null
"""0105540001""","""La Luna""",15540001,"""Bgy""",null,null,null,"""R""",903,null,null
"""0701225032""","""Tubod Mar""",71225032,"""Bgy""",null,null,null,"""R""",663,null,null
"""0806417042""","""Santa Maria""",86417042,"""Bgy""",null,null,null,"""R""",209,null,null
"""0102912023""","""Miramar""",12912023,"""Bgy""",null,null,null,"""R""",1449,null,null


## col: `correspondence_code`

In [7]:
# cast to string because right now they're i64
renamed_df = renamed_df.with_columns(
    pl.col("correspondence_code").cast(pl.Utf8)
)

In [8]:
# make sure all are 9 chars or 0 if it's empty
renamed_df = renamed_df.with_columns(pl.col("correspondence_code").fill_null(""))
renamed_df = renamed_df.with_columns(
    pl.when(pl.col("correspondence_code").str.len_chars() == 8)
    .then(pl.col("correspondence_code").str.zfill(9))
    .otherwise(pl.col("correspondence_code")),
)

In [9]:
renamed_df.sample(10)

psgc_id,psgc_name,correspondence_code,geographic_level,old_name,city_class,income_classification,settlement_type,population,remarks,status
str,str,str,str,str,str,str,str,i64,str,str
"""0500517008""","""Bogñabong""","""050517008""","""Bgy""",null,null,null,"""R""",3054,null,null
"""0304922019""","""Santa Rita""","""034922019""","""Bgy""",null,null,null,"""R""",4083,null,null
"""0701243001""","""Bagacay""","""071243001""","""Bgy""",null,null,null,"""R""",4013,null,null
"""0600411008""","""Calangcang""","""060411008""","""Bgy""",null,null,null,"""R""",2161,null,null
"""0301410049""","""Santo Niño ""","""031410049""","""Bgy""",null,null,null,"""U""",661,null,"""Pob."""
"""0306909034""","""Tubectubang""","""036909034""","""Bgy""",null,null,null,"""R""",2796,null,null
"""0806009005""","""Candayao""","""086009005""","""Bgy""",null,null,null,"""R""",340,null,null
"""1380611011""","""Barangay 677""","""133911011""","""Bgy""",null,null,null,"""U""",2235,null,null
"""0803734025""","""Santa Paz""","""083734025""","""Bgy""",null,null,null,"""R""",1739,null,null


## col: `settlement_type`

In [10]:
# map to the valid enum
SettlementTypeEnum = pl.Enum(
    categories=[
        "urban",
        "rural",
        "-",
    ]
)
settlement_type_map = {"R": "rural", "U": "urban", "": None}
renamed_df = renamed_df.with_columns(
    pl.col("settlement_type").replace(settlement_type_map)
)
renamed_df = renamed_df.with_columns(pl.col("settlement_type").cast(SettlementTypeEnum))
renamed_df.sample(10)

psgc_id,psgc_name,correspondence_code,geographic_level,old_name,city_class,income_classification,settlement_type,population,remarks,status
str,str,str,str,str,str,str,enum,i64,str,str
"""0702207001""","""Alawijao""","""072207001""","""Bgy""",null,null,null,"""rural""",1088,null,null
"""0501716031""","""Santa Isabel""","""051716031""","""Bgy""",null,null,null,"""rural""",654,null,null
"""0702220009""","""Gilutongan""","""072220009""","""Bgy""",null,null,null,"""rural""",1851,null,null
"""0603046013""","""Batga""","""063046013""","""Bgy""",null,null,null,"""rural""",267,null,null
"""0306916003""","""Alvindia Segundo""","""036916003""","""Bgy""",null,null,null,"""rural""",1960,null,null
"""0501719032""","""San Jose""","""051719032""","""Bgy""",null,null,null,"""rural""",1521,null,null
"""1030900000""","""City of Iligan""","""103504000""","""City""",null,"""HUC""","""1st""",null,368132,null,null
"""0307703009""","""Masagana ""","""037703009""","""Bgy""",null,null,null,"""rural""",2305,null,"""Pob."""
"""0702249019""","""Villahermosa""","""072249019""","""Bgy""",null,null,null,"""rural""",621,null,null


## col: `status`

In [11]:
renamed_df["status"].value_counts()

status,count
str,u32
"""Capital""",82
"""Pob.""",2773
null,40914


In [12]:
BarangayStatusEnum = pl.Enum(
    categories=[
        "poblacion",
        "capital",
    ]
)
barangay_status_map = {
    "Pob.": "poblacion",
    "Capital": "capital",
}
renamed_df = renamed_df.with_columns(pl.col("status").replace(barangay_status_map))
renamed_df = renamed_df.with_columns(pl.col("status").cast(BarangayStatusEnum))
renamed_df.sample(10)

psgc_id,psgc_name,correspondence_code,geographic_level,old_name,city_class,income_classification,settlement_type,population,remarks,status
str,str,str,str,str,str,str,enum,i64,str,enum
"""0306906005""","""Ayson""","""036906005""","""Bgy""",null,null,null,"""rural""",1956,null,null
"""0906615014""","""Tinutungan""","""156615014""","""Bgy""",null,null,null,"""rural""",2146,null,null
"""1108204026""","""Poblacion""","""118204026""","""Bgy""",null,null,null,"""urban""",8784,null,null
"""1004213028""","""El Paraiso""","""104213028""","""Bgy""",null,null,null,"""rural""",646,null,null
"""1600312000""","""Trento""","""160312000""","""Mun""",null,null,"""1st""",null,51179,null,null
"""0203106015""","""Ngarag""","""023106015""","""Bgy""",null,null,null,"""rural""",1206,null,null
"""0403404019""","""Barangay Tres ""","""043404019""","""Bgy""",null,null,null,"""urban""",2216,null,"""poblacion"""
"""0803719020""","""Liberty""","""083719020""","""Bgy""",null,null,null,"""rural""",3158,null,null
"""1903610017""","""Malna Proper""","""153610017""","""Bgy""",null,null,null,"""rural""",1245,null,null


## col: `old_name`, `remarks`

In [13]:
renamed_df = renamed_df.with_columns(
    [
        pl.col("old_name").fill_null(""),
        pl.col("remarks").fill_null(""),
    ]
)
renamed_df.sample(5)

psgc_id,psgc_name,correspondence_code,geographic_level,old_name,city_class,income_classification,settlement_type,population,remarks,status
str,str,str,str,str,str,str,enum,i64,str,enum
"""0203122014""","""Calangigan""","""023122014""","""Bgy""","""Calamagui""",null,null,"""rural""",833,"""""",null
"""0405605032""","""San Isidro Ilaya""","""045605032""","""Bgy""","""""",null,null,"""rural""",487,"""""",null
"""0600406029""","""San Isidro""","""060406029""","""Bgy""","""""",null,null,"""rural""",1985,"""""",null
"""1403213025""","""Nambaran""","""143213025""","""Bgy""","""""",null,null,"""rural""",4419,"""""",null
"""0102820010""","""San Guillermo""","""012820010""","""Bgy""","""""",null,null,"""urban""",1887,"""""",null


## col: `city_class`

In [14]:
renamed_df.select("city_class").unique()

city_class
str
"""CC"""
null
"""HUC"""
"""ICC"""


In [15]:
CityClassEnum = pl.Enum(
    categories=[
        "highly_urbanized_city",
        "independent_component_city",
        "component_city",
    ]
)
city_class_map = {
    "HUC": "highly_urbanized_city",
    "ICC": "independent_component_city",
    "CC": "component_city",
}
renamed_df = renamed_df.with_columns(pl.col("city_class").replace(city_class_map))
renamed_df = renamed_df.with_columns(pl.col("city_class").cast(CityClassEnum))
renamed_df.sample(10)

psgc_id,psgc_name,correspondence_code,geographic_level,old_name,city_class,income_classification,settlement_type,population,remarks,status
str,str,str,str,str,enum,str,enum,i64,str,enum
"""0906601028""","""Panglima Misuari""","""156601028""","""Bgy""","""Sasak""",null,null,"""rural""",1625,"""""",null
"""1806106029""","""Pasihagon""","""076106029""","""Bgy""","""""",null,null,"""rural""",1306,"""""",null
"""0205013006""","""Dadap""","""025013006""","""Bgy""","""""",null,null,"""rural""",1409,"""""",null
"""1102414009""","""Laperas""","""112414009""","""Bgy""","""""",null,null,"""rural""",713,"""""",null
"""0806007009""","""Burabod II ""","""086007009""","""Bgy""","""""",null,null,"""rural""",1168,"""""","""poblacion"""
"""1704003018""","""Matandang Gasan""","""174003018""","""Bgy""","""""",null,null,"""rural""",1833,"""""",null
"""0803714001""","""Balucanad""","""083714001""","""Bgy""","""""",null,null,"""rural""",2464,"""""",null
"""0603040017""","""Cata-an""","""063040017""","""Bgy""","""""",null,null,"""rural""",1276,"""""",null
"""1804503001""","""Amontay""","""064503001""","""Bgy""","""""",null,null,"""rural""",2957,"""""",null


In [16]:
renamed_df.select("city_class").unique()

city_class
enum
"""highly_urbanized_city"""
null
"""independent_component_city"""
"""component_city"""


## col: `income_classification`

In [17]:
renamed_df.select("income_classification").unique().to_series().to_list()

['1st', '4th', None, '2nd', '4th*', '3rd*', '5th', '3rd', '', '-', '2nd*']

In [18]:
IncomeClassificationEnum = pl.Enum(
    categories=[
        "1st",
        "1st*",
        "2nd",
        "2nd*",
        "3rd",
        "3rd*",
        "4th",
        "4th*",
        "5th",
        "5th*",
        "-",
    ]
)
income_classification_map = {
    "1st": "1st",
    "1st*": "1st*",
    "2nd": "2nd",
    "2nd*": "2nd*",
    "3rd": "3rd",
    "3rd*": "3rd*",
    "4th": "4th",
    "4th*": "4th*",
    "5th": "5th",
    "5th*": "5th*",
    "": None,
    "-": "-",
}
renamed_df = renamed_df.with_columns(
    pl.col("income_classification").replace(income_classification_map)
)
renamed_df = renamed_df.with_columns(
    pl.col("income_classification").cast(IncomeClassificationEnum)
)
renamed_df.sample(10)

psgc_id,psgc_name,correspondence_code,geographic_level,old_name,city_class,income_classification,settlement_type,population,remarks,status
str,str,str,str,str,enum,enum,enum,i64,str,enum
"""0931700074""","""Santa Barbara""","""097332074""","""Bgy""","""""",null,null,"""urban""",7090,"""""",null
"""0500501017""","""Hindi""","""050501017""","""Bgy""","""""",null,null,"""rural""",4439,"""""",null
"""1102401015""","""Managa""","""112401015""","""Bgy""","""""",null,null,"""rural""",5185,"""""",null
"""0105511023""","""Cadre Site""","""015511023""","""Bgy""","""""",null,null,"""rural""",2776,"""""",null
"""0501603018""","""Magang""","""051603018""","""Bgy""","""""",null,null,"""urban""",6184,"""""",null
"""0403408009""","""Calumpang ""","""043408009""","""Bgy""","""""",null,null,"""rural""",418,"""""","""poblacion"""
"""1230800033""","""Calumpang""","""126303033""","""Bgy""","""""",null,null,"""urban""",90480,"""""",null
"""0831600133""","""Barangay 99""","""083747133""","""Bgy""","""Diit""",null,null,"""urban""",6866,"""""",null
"""0405627026""","""Remedios II""","""045627026""","""Bgy""","""""",null,null,"""rural""",173,"""""",null


In [19]:
IncomeClassificationCleanEnum = pl.Enum(
    categories=[
        "1st",
        "2nd",
        "3rd",
        "4th",
        "5th",
    ]
)
income_classification_clean_map = {
    "1st": "1st",
    "1st*": "1st",
    "2nd": "2nd",
    "2nd*": "2nd",
    "3rd": "3rd",
    "3rd*": "3rd",
    "4th": "4th",
    "4th*": "4th",
    "5th": "5th",
    "5th*": "5th",
    "": None,
    "-": None,
}
renamed_df = renamed_df.with_columns(
    pl.col("income_classification")
    .cast(pl.String)
    .replace(income_classification_clean_map)
    .alias("income_classification_clean")
)
renamed_df = renamed_df.with_columns(
    pl.col("income_classification_clean")
    .cast(IncomeClassificationCleanEnum)
    .alias("income_classification_clean")
)
renamed_df.sample(10)

psgc_id,psgc_name,correspondence_code,geographic_level,old_name,city_class,income_classification,settlement_type,population,remarks,status,income_classification_clean
str,str,str,str,str,enum,enum,enum,i64,str,enum,enum
"""1705313007""","""Nangalao""","""175313007""","""Bgy""","""""",null,null,"""rural""",2517,"""""",null,null
"""0103307033""","""Quinavite""","""013307033""","""Bgy""","""""",null,null,"""rural""",3687,"""""",null,null
"""0600613024""","""Maybato Sur""","""060613024""","""Bgy""","""""",null,null,"""rural""",2320,"""""",null,null
"""0603026008""","""Camangay""","""063026008""","""Bgy""","""""",null,null,"""rural""",877,"""""",null,null
"""1380500019""","""New Zañiga""","""137401019""","""Bgy""","""""",null,null,"""urban""",8872,"""""",null,null
"""0103309017""","""Urayong""","""013309017""","""Bgy""","""""",null,null,"""rural""",706,"""""",null,null
"""0105511050""","""Pantol""","""015511050""","""Bgy""","""""",null,null,"""rural""",1342,"""""",null,null
"""1400126000""","""Tubo""","""140126000""","""Mun""","""""",null,"""2nd""",null,4941,"""""",null,"""2nd"""
"""0405628005""","""Bagupaye""","""045628005""","""Bgy""","""""",null,null,"""rural""",2224,"""""",null,null


## Checking Levels

In [20]:
sorted(
    [
        x
        for x in renamed_df.select("geographic_level").unique().to_series().to_list()
        if x
    ]
)

['Bgy', 'City', 'Mun', 'Prov', 'Reg', 'SubMun']

In [21]:
barangay = renamed_df.filter(pl.col("geographic_level") == "Bgy")
barangay = barangay.drop(
    [
        "city_class",
        "income_classification",
        "income_classification_clean",
        "geographic_level",
    ],
    strict=False,
)

In [22]:
municipality = renamed_df.filter(pl.col("geographic_level") == "Mun")
municipality = municipality.drop(
    [
        "city_class",
        "geographic_level",
        "settlement_type",
    ],
    strict=False,
)

In [23]:
city = renamed_df.filter(pl.col("geographic_level") == "City")
city = city.drop(
    [
        "geographic_level",
        "settlement_type",
    ],
    strict=False,
)

In [24]:
province = renamed_df.filter(pl.col("geographic_level") == "Prov")
province = province.drop(
    [
        "geographic_level",
        "settlement_type",
        "status",
        "city_class",
    ],
    strict=False,
)

In [25]:
region = renamed_df.filter(pl.col("geographic_level") == "Reg")
region = region.drop(
    [
        "income_classification_clean",
        "income_classification",
        "status",
        "settlement_type",
        "city_class",
        "geographic_level",
    ],
    strict=False,
)

In [26]:
region.sample(5)

psgc_id,psgc_name,correspondence_code,old_name,population,remarks
str,str,str,str,i64,str
"""0500000000""","""Region V (Bicol Region)""","""050000000""","""""",6064426,""""""
"""0700000000""","""Region VII (Central Visayas)""","""070000000""","""""",6640875,""""""
"""1600000000""","""Region XIII (Caraga)""","""160000000""","""""",2865196,""""""
"""0100000000""","""Region I (Ilocos Region)""","""010000000""","""""",5342453,""""""
"""1300000000""","""National Capital Region (NCR)""","""130000000""","""""",14001751,""""""


In [27]:
submunicipality = renamed_df.filter(pl.col("geographic_level") == "SubMun")
submunicipality = submunicipality.drop(
    [
        "city_class",
        "geographic_level",
        "settlement_type",
        "income_classification",
        "income_classification_clean",
        "status",
    ],
    strict=False,
)
submunicipality.sample(10)

psgc_id,psgc_name,correspondence_code,old_name,population,remarks
str,str,str,str,i64,str
"""1380608000""","""Ermita""","""133908000""","""""",22863,""""""
"""1380613000""","""Port Area""","""133913000""","""""",76783,""""""
"""1380603000""","""Quiapo""","""133903000""","""""",32236,""""""
"""1380604000""","""San Nicolas""","""133904000""","""""",46350,""""""
"""1380605000""","""Santa Cruz""","""133905000""","""""",134484,""""""
"""1380609000""","""Intramuros""","""133909000""","""""",7437,""""""
"""1380602000""","""Binondo""","""133902000""","""""",23935,""""""
"""1380612000""","""Pandacan""","""133912000""","""""",90194,""""""
"""1380614000""","""Santa Ana""","""133914000""","""""",208117,""""""


In [28]:
dfs = {
    "barangay": barangay,
    "municipality": municipality,
    "submunicipality": submunicipality,
    "city": city,
    "province": province,
    "region": region,
}

In [29]:
raise KeyboardInterrupt

KeyboardInterrupt: 

## Building out the Junction Table

In [30]:
from IPython.display import display

semantic_code_meaning = {
    "1st_level": 10,
    "2nd_level": 7,
    "3rd_level": 5,
    "4th_level": 2,
}
junction_table = barangay
for level, nth_char in semantic_code_meaning.items():
    junction_table = junction_table.with_columns(
        (
            pl.col("psgc_id").str.slice(0, nth_char).cast(str)
            + pl.lit("0").repeat_by(pl.lit(10 - nth_char)).list.join("")
        ).alias(level)
    )

junction_table = junction_table.with_columns(pl.col("psgc_id").alias("1st_level"))
junction_table = junction_table.select(
    ["1st_level", "2nd_level", "3rd_level", "4th_level"]
)

In [31]:
expanded_junction = junction_table.select(
    ["1st_level", "2nd_level", "3rd_level", "4th_level"]
)
selection_columns = ["1st_level", "2nd_level", "3rd_level", "4th_level"]
for identifier in semantic_code_meaning:
    for division, division_df in dfs.items():
        division_id = division + "_" + identifier
        expanded_junction = expanded_junction.join(
            division_df, how="full", left_on=identifier, right_on="psgc_id"
        )
        expanded_junction = expanded_junction.with_columns(
            pl.col("psgc_name").alias(f"{division_id}_name"),
            pl.col("psgc_id").alias(division_id),
        )
        selection_columns.append(division_id)
        selection_columns.append(f"{division_id}_name")
        expanded_junction = expanded_junction.select(selection_columns)

columns_to_select = []
for division, _ in dfs.items():

    layers = []
    layers_name = []
    for level in semantic_code_meaning.keys():
        layers.append(f"{division}_{level}")
        layers_name.append(f"{division}_{level}_name")

    print(f"For division: {division}")
    print(layers)
    print(layers_name)
    print()
    expanded_junction = expanded_junction.with_columns(
        pl.coalesce([pl.col(_col) for _col in layers_name]).alias(f"{division}_name"),
        pl.coalesce([pl.col(_col) for _col in layers]).alias(f"{division}")
    )
    columns_to_select.append(division)
    columns_to_select.append(f"{division}_name")

expanded_junction.select(columns_to_select).sample(10)

For division: barangay
['barangay_1st_level', 'barangay_2nd_level', 'barangay_3rd_level', 'barangay_4th_level']
['barangay_1st_level_name', 'barangay_2nd_level_name', 'barangay_3rd_level_name', 'barangay_4th_level_name']

For division: municipality
['municipality_1st_level', 'municipality_2nd_level', 'municipality_3rd_level', 'municipality_4th_level']
['municipality_1st_level_name', 'municipality_2nd_level_name', 'municipality_3rd_level_name', 'municipality_4th_level_name']

For division: submunicipality
['submunicipality_1st_level', 'submunicipality_2nd_level', 'submunicipality_3rd_level', 'submunicipality_4th_level']
['submunicipality_1st_level_name', 'submunicipality_2nd_level_name', 'submunicipality_3rd_level_name', 'submunicipality_4th_level_name']

For division: city
['city_1st_level', 'city_2nd_level', 'city_3rd_level', 'city_4th_level']
['city_1st_level_name', 'city_2nd_level_name', 'city_3rd_level_name', 'city_4th_level_name']

For division: province
['province_1st_level', 'pr

barangay,barangay_name,municipality,municipality_name,submunicipality,submunicipality_name,city,city_name,province,province_name,region,region_name
str,str,str,str,str,str,str,str,str,str,str,str
"""0504105002""","""Aguada""",null,null,null,null,null,null,null,null,null,null
"""0603045051""","""Tan Pael""",null,null,null,null,null,null,null,null,null,null
"""1204710027""","""New Lawa-an""",null,null,null,null,null,null,null,null,null,null
"""0402119037""","""Tolentino East""",null,null,null,null,"""0402119000""","""City of Tagaytay""","""0402100000""","""Cavite""","""0400000000""","""Region IV-A (CALABARZON)"""
"""0504121023""","""Mabuhay""","""0504121000""","""Uson""",null,null,null,null,"""0504100000""","""Masbate""","""0500000000""","""Region V (Bicol Region)"""
"""0603012038""","""Morubuan""","""0603012000""","""Cabatuan""",null,null,null,null,"""0603000000""","""Iloilo""","""0600000000""","""Region VI (Western Visayas)"""
"""0702243020""","""Poblacion""",null,null,null,null,null,null,null,null,null,null
"""0402110021""","""Mahabangkahoy Lejos""",null,null,null,null,null,null,null,null,null,null
"""0906613036""","""Lumping Pigih Daho""",null,null,null,null,null,null,null,null,null,null


In [32]:
lightweight_junction = expanded_junction.select(columns_to_select)

In [33]:
lightweight_junction = lightweight_junction.with_columns(
    nulls=pl.sum_horizontal(lightweight_junction.select(pl.all().is_null()))
)

In [34]:
lightweight_junction.select(pl.col("nulls").value_counts())


nulls
struct[2]
"{4,39775}"
"{6,2236}"
"{10,131269}"


In [35]:
final_junction_table = lightweight_junction.filter(pl.col("nulls").is_in([6,4])).select(columns_to_select)

In [36]:
final_junction_table

barangay,barangay_name,municipality,municipality_name,submunicipality,submunicipality_name,city,city_name,province,province_name,region,region_name
str,str,str,str,str,str,str,str,str,str,str,str
"""1380100001""","""Barangay 1""",null,null,null,null,"""1380100000""","""City of Caloocan""",null,null,"""1300000000""","""National Capital Region (NCR)"""
"""1380100002""","""Barangay 2""",null,null,null,null,"""1380100000""","""City of Caloocan""",null,null,"""1300000000""","""National Capital Region (NCR)"""
"""1380100003""","""Barangay 3""",null,null,null,null,"""1380100000""","""City of Caloocan""",null,null,"""1300000000""","""National Capital Region (NCR)"""
"""1380100004""","""Barangay 4""",null,null,null,null,"""1380100000""","""City of Caloocan""",null,null,"""1300000000""","""National Capital Region (NCR)"""
"""1380100005""","""Barangay 5""",null,null,null,null,"""1380100000""","""City of Caloocan""",null,null,"""1300000000""","""National Capital Region (NCR)"""
…,…,…,…,…,…,…,…,…,…,…,…
"""1999908006""","""Manaulanan""","""1999908000""","""Tugunan""",null,null,null,null,null,null,"""1900000000""","""Bangsamoro Autonomous Region I…"
"""1999908007""","""Pamalian""","""1999908000""","""Tugunan""",null,null,null,null,null,null,"""1900000000""","""Bangsamoro Autonomous Region I…"
"""1999908008""","""Tapodoc""","""1999908000""","""Tugunan""",null,null,null,null,null,null,"""1900000000""","""Bangsamoro Autonomous Region I…"


# 🥈 Silver

# 🥈Silver -> 🥇 Gold

For Silver to Gold, we now have to detect **changes**

In [37]:
IDENTITY_COLUMNS = [
    "psgc_id",
    "psgc_name",
]
AUX_COLUMNS = [
    "field_hash",
    "identity_hash",
    "ingestion_datetime",
    "valid_from",
    "valid_to",
    "surrogate_id",
    "durable_id",
    "version",
]

In [38]:
import blake3
from typing import Any, Dict

tables: Dict[str, pl.DataFrame] = {}

for name, df in dfs.items():
    field_columns = [
        col for col in dfs[name].columns if col not in (IDENTITY_COLUMNS + AUX_COLUMNS)
    ]

    # Filling out the valid_from date
    release_date = pl.Series([metadata.valid_from]).str.strptime(
        pl.Datetime, "%Y-%m-%dT%H:%M:%S%.3f%z"
    )[0]
    dfs[name] = dfs[name].with_columns(
        pl.lit(release_date).alias("valid_from"),
    )

    # Creating fact tables
    tables["fact_population_" + name] = dfs[name].select(
        [
            "psgc_id",
            "psgc_name",
            "population",
            "valid_from",
        ]
    )

    # Creating dimension tables
    tables["dim_" + name] = dfs[name].drop("population")

    # creating the identity_hash and fields_hash columns
    for table_type in ["fact_population_", "dim_"]:
        tables[table_type + name] = tables[table_type + name].with_columns(
            pl.struct(IDENTITY_COLUMNS)
            .map_elements(
                lambda row: blake3.blake3(
                    f"{row["psgc_name"]}_{row["psgc_id"]}".encode(encoding="utf-8")
                ).hexdigest()
            )
            .alias("identity_hash")
        )

        field_columns = [
            col
            for col in tables[table_type + name].columns
            if col not in (IDENTITY_COLUMNS + AUX_COLUMNS)
        ]

        tables[table_type + name] = tables[table_type + name].with_columns(
            pl.struct(field_columns)
            .map_elements(
                lambda row: blake3.blake3(
                    "_".join([str(row[field]) for field in field_columns]).encode(
                        encoding="utf-8"
                    )
                ).hexdigest(),
                return_dtype=pl.String,
            )
            .alias("fields_hash")
        )

        tables[table_type + name] = tables[table_type + name].with_columns(
            pl.lit(dt.datetime.now(tz=dt.timezone.utc)).alias("ingestion_datetime"),
            pl.Series(
                name="surrogate_id",
                values=[str(uuid4()) for _ in range(len(tables[table_type + name]))],
            )
            .cast(pl.String)
            .alias("surrogate_id"),
        )
        tables[table_type + name] = tables[table_type + name].select(
            [
                "surrogate_id",
                "ingestion_datetime",
                *IDENTITY_COLUMNS,
                *field_columns,
                "identity_hash",
                "fields_hash",
                "valid_from",
            ]
        )       

SchemaError: invalid series dtype: expected `String`, got `datetime[μs, UTC]` for series with name ``

In [ ]:
from pydantic import BaseModel
from zoneinfo import ZoneInfo


class DataValiditySet3DataFrames(BaseModel):
    y_valid_from: dt.datetime
    y_valid_to: dt.datetime
    x_valid_from: dt.datetime | None = None
    x_valid_to: dt.datetime | None = None
    z_valid_from: dt.datetime | None = None
    z_valid_to: dt.datetime | None = None


def generate_uuid_column(df: pl.DataFrame, name: str) -> pl.DataFrame:
    """
    Generate uuid column for all rows in df
    """
    new_df = df.with_columns(
        pl.Series(
            name=name,
            values=[str(uuid4()) for _ in range(len(df))],
        )
        .cast(pl.String)
        .alias(name),
    )
    return new_df


def generate_blake3_hash(struct: Dict[str, str]):
    strings = [val or "" for _, val in struct.items()]
    return blake3.blake3("_".join(strings).encode(encoding="utf8")).hexdigest()


def column_hash(
    df: pl.DataFrame, columns_to_hash: List[str], hash_name: str
) -> pl.DataFrame:
    """
    Hash a column based on provided columns to hash and save it as a column with the
    name `{hash_name}_hash`
    """

    new_df = df.with_columns(
        pl.struct(columns_to_hash)
        .map_elements(lambda s: generate_blake3_hash(s))
        .alias(hash_name)
    )
    return new_df


def add_ingestion_datetime(df: pl.DataFrame) -> pl.DataFrame:
    new_df = df.with_columns(
        pl.lit(dt.datetime.now(tz=dt.timezone.utc)).alias("ingestion_datetime")
    )
    return new_df


def compute_validity(
    y_time: dt.datetime,
    x_time: dt.datetime | None = None,
    z_time: dt.datetime | None = None,
) -> DataValiditySet3DataFrames:
    y_valid_from = y_time
    if x_time is None:
        x_valid_from = None
        x_valid_to = None
    if x_time is not None:
        x_valid_from = x_time
        x_valid_to = y_time - dt.timedelta(seconds=1)
    if z_time is None:
        y_valid_to = dt.datetime(
            year=9999, month=12, day=31, tzinfo=ZoneInfo("Asia/Manila")
        )
        z_valid_from = None
        z_valid_to = None
    if z_time is not None:
        y_valid_to = z_time - dt.timedelta(seconds=1)
        z_valid_from = z_time
        z_valid_to = dt.datetime(
            year=9999, month=12, day=31, tzinfo=ZoneInfo("Asia/Manila")
        )
    data = DataValiditySet3DataFrames(
        x_valid_from=x_valid_from,
        x_valid_to=x_valid_to,
        y_valid_from=y_valid_from,
        y_valid_to=y_valid_to,
        z_valid_from=z_valid_from,
        z_valid_to=z_valid_to,
    )
    return data


class DataChangeSet:
    """
    x < y < z

    - y = data to be inserted
    - x = previous data point, can be None (meaning y is the first data)
    - z = next data point, can be None (meaning y is the latest)
    """

    def __init__(
        self,
        *,
        x_df: pl.DataFrame | None,
        y_df: pl.DataFrame,
        z_df: pl.DataFrame | None,
    ):
        y_df = column_hash(
            y_df, columns_to_hash=IDENTITY_COLUMNS, hash_name="identity_hash"
        )
        fields_to_hash = [
            col for col in y_df.columns if col not in IDENTITY_COLUMNS + AUX_COLUMNS
        ]
        y_df = column_hash(
            y_df, columns_to_hash=fields_to_hash, hash_name="fields_hash"
        )
        y_df = add_ingestion_datetime(y_df)
        y_df = generate_uuid_column(y_df, name="surrogate_id")
        if x_df is None and z_df is None:
            y_df = generate_uuid_column(y_df, name="durable_id")
            y_df = y_df.with_columns(
                [
                    pl.lit(0, dtype=pl.UInt16).alias("version"),
                    pl.col("valid_from")
                    .map_elements(
                        lambda s: compute_validity(y_time=s).y_valid_to,
                        return_dtype=pl.Datetime(time_zone="Asia/Manila"),
                    )
                    .alias("valid_to"),
                ]
            )

        # We're now going to check if there are any deltas we need to update
        if x_df is not None:
            # --------------------------------------------------------------------------
            # Dataset with no changes
            # --------------------------------------------------------------------------
            no_changes = x_df.join(
                y_df, on=["identity_hash", "fields_hash"], how="inner"
            )
            changes_in_x = x_df.filter(
                ~pl.col("identity_hash").is_in(
                    no_changes.get_column("identity_hash").implode()
                )
            )
            changes_in_y = y_df.filter(
                ~pl.col("identity_hash").is_in(
                    no_changes.get_column("identity_hash").implode()
                )
            )

            durable_ids_with_no_changes = no_changes.get_column("durable_id")
            dataset_with_no_updates = x_df.filter(
                pl.col("durable_id").is_in(durable_ids_with_no_changes.implode())
            )

            # --------------------------------------------------------------------------
            # Dataset with field changes
            # --------------------------------------------------------------------------
            # This is gonna yield x2 rows for each change. One for the change that
            # happened, and another for updating the previous row.
            # --------------------------------------------------------------------------
            # Retrieve data in y that has the same identity in x BUT (implied) has a
            # different field hash.
            updated_fields_in_y = changes_in_y.join(
                changes_in_x, on=["identity_hash"], how="inner"
            )

            # Selecting relevant columns
            columns_to_select_in_updated_fields_in_y = [
                col
                for col in updated_fields_in_y.columns
                if col in IDENTITY_COLUMNS + AUX_COLUMNS + fields_to_hash
            ]
            dataset_with_field_updates = updated_fields_in_y.select(
                columns_to_select_in_updated_fields_in_y
            )

            # Generating the new fields hash
            dataset_with_field_updates = column_hash(
                dataset_with_field_updates,
                columns_to_hash=fields_to_hash,
                hash_name="fields_hash",
            )

            # Pause, before creating the official "updates", let's look back first
            # towards the old record and update the valid_to and version
            changes_in_x.filter(pl.col)

            # One-upping the version
            dataset_with_field_updates = dataset_with_field_updates.with_columns(
                (pl.col("version") + pl.lit(1)).alias("version")
            )

            # reordering columns based on previous
            dataset_with_field_updates = dataset_with_field_updates.select(x_df.columns)

            self.no_changes = no_changes
            self.changes_in_x = changes_in_x
            self.changes_in_y = changes_in_y
            self.updated_fields_in_y = updated_fields_in_y

            self.durable_ids_with_no_changes = durable_ids_with_no_changes
            self.dataset_with_no_updates = dataset_with_no_updates
            self.dataset_with_field_updates = dataset_with_field_updates

        self.x_df = x_df
        self.y_df = y_df
        self.z_df = z_df

In [128]:
from zoneinfo import ZoneInfo

tzinfo = ZoneInfo("Asia/Manila")
date_x = dt.datetime(year=2024, month=1, day=1, tzinfo=tzinfo)
date_y = dt.datetime(year=2025, month=1, day=1, tzinfo=tzinfo)
date_z = dt.datetime(year=2026, month=1, day=1, tzinfo=tzinfo)

sample_x = pl.DataFrame(
    data={
        "psgc_id": ["0001", "0002", "0003", "0004"],
        "psgc_name": ["Barangay1", "Barangay2", "Barangay3", "Barangay4"],
        "settlement_type": ["urban", "urban", "rural", "rural"],
        "status": [None, "poblacion", None, None],
        "valid_from": [date_x, date_x, date_x, date_x],
    }
)

sample_y = pl.DataFrame(
    data={
        "psgc_id": ["0001", "0002", "0003", "0005"],
        "psgc_name": ["Barangay1", "Barangay2", "Barangay3-A", "Barangay4"],
        "settlement_type": ["urban", "rural", "rural", "rural"],
        "status": [None, "poblacion", None, None],
        "valid_from": [date_y, date_y, date_y, date_y],
    }
)


sample_z = pl.DataFrame(
    data={
        "psgc_id": ["0001", "0002", "0003", "0004"],
        "psgc_name": ["Barangay1", "Barangay2", "Barangay3", "Barangay4"],
        "settlement_type": ["urban", "urban", "rural", "rural"],
        "status": [None, "poblacion", None, None],
        "valid_from": [date_z, date_z, date_z, date_z],
    }
)


first_df = DataChangeSet(x_df=None, y_df=sample_x, z_df=None)
with_prev = DataChangeSet(x_df=first_df.y_df, y_df=sample_y, z_df=None)

In [132]:
with_prev.changes_in_x.filter(pl.col("durable_id").is_in(with_prev.updated_fields_in_y.get_column("durable_id").implode()))

psgc_id,psgc_name,settlement_type,status,valid_from,identity_hash,fields_hash,ingestion_datetime,surrogate_id,durable_id,version,valid_to
str,str,str,str,"datetime[μs, Asia/Manila]",str,str,"datetime[μs, UTC]",str,str,u16,"datetime[μs, Asia/Manila]"
"""0002""","""Barangay2""","""urban""","""poblacion""",2024-01-01 00:00:00 PST,"""7c42c5fb78f5c1861a0276de94f6ae…","""cb52d0bd57a59bcc88133d10efb5da…",2025-11-09 18:56:40.887835 UTC,"""da19db3b-3a4e-4409-ac5d-e69f57…","""b7ab2f83-3cf1-42ed-b9bb-662605…",0,9999-12-31 00:00:00 PST


In [113]:
print("with no updates")
display(with_prev.dataset_with_no_updates)
print("with_field_updates")
display(with_prev.dataset_with_field_updates)

with no updates


psgc_id,psgc_name,settlement_type,status,valid_from,identity_hash,fields_hash,ingestion_datetime,surrogate_id,durable_id,version
str,str,str,str,"datetime[μs, Asia/Manila]",str,str,"datetime[μs, UTC]",str,str,u16
"""0001""","""Barangay1""","""urban""",null,2024-01-01 00:00:00 PST,"""b9da21229fee23d248548c8387f578…","""808b40e0b6c1555796fdde1a408599…",2025-11-09 18:38:57.140120 UTC,"""783914fe-a603-40a6-b5ba-e97ae5…","""43ce55ae-5942-4e88-a41f-7cc624…",0


with_field_updates


psgc_id,psgc_name,settlement_type,status,valid_from,identity_hash,fields_hash,ingestion_datetime,surrogate_id,durable_id,version
str,str,str,str,"datetime[μs, Asia/Manila]",str,str,"datetime[μs, UTC]",str,str,u16
"""0002""","""Barangay2""","""rural""","""poblacion""",2025-01-01 00:00:00 PST,"""7c42c5fb78f5c1861a0276de94f6ae…","""2ed2f2fb3470a0e8c9d1da988e6b30…",2025-11-09 18:38:57.141947 UTC,"""c768cc5b-6a63-4695-9155-5e1996…","""37fa0a65-0fcd-472d-92f7-255cec…",1


In [95]:
with_prev.durable_ids_with_no_changes

durable_id
str
"""172a4342-ed96-4656-a579-826a4f…"


In [96]:
with_prev.changes_in_y

psgc_id,psgc_name,settlement_type,status,valid_from,identity_hash,fields_hash,ingestion_datetime,surrogate_id
str,str,str,str,"datetime[μs, Asia/Manila]",str,str,"datetime[μs, UTC]",str
"""0002""","""Barangay2""","""rural""","""poblacion""",2025-01-01 00:00:00 PST,"""7c42c5fb78f5c1861a0276de94f6ae…","""2ed2f2fb3470a0e8c9d1da988e6b30…",2025-11-09 18:21:20.431038 UTC,"""6fe57ef5-15ee-4eb5-997a-8b2a29…"
"""0003""","""Barangay3-A""","""rural""",null,2025-01-01 00:00:00 PST,"""a4018606523d08c8452a86865440e5…","""2c4e89dc8fd02af69254564d713b17…",2025-11-09 18:21:20.431038 UTC,"""5956366c-f3f8-4160-8ce9-4f124f…"
"""0005""","""Barangay4""","""rural""",null,2025-01-01 00:00:00 PST,"""07e0f4a38fb459156f4a404776b549…","""2c4e89dc8fd02af69254564d713b17…",2025-11-09 18:21:20.431038 UTC,"""f46a31e2-db6f-4f92-b7a9-2cc5c6…"


In [97]:
with_prev.changes_in_x

psgc_id,psgc_name,settlement_type,status,valid_from,identity_hash,fields_hash,ingestion_datetime,surrogate_id,durable_id,version
str,str,str,str,"datetime[μs, Asia/Manila]",str,str,"datetime[μs, UTC]",str,str,u16
"""0002""","""Barangay2""","""urban""","""poblacion""",2024-01-01 00:00:00 PST,"""7c42c5fb78f5c1861a0276de94f6ae…","""cb52d0bd57a59bcc88133d10efb5da…",2025-11-09 18:21:20.429501 UTC,"""70b72584-6ddf-4e2e-8cf8-ed2852…","""d62228dd-67a0-4fa1-90a8-63c73c…",0
"""0003""","""Barangay3""","""rural""",null,2024-01-01 00:00:00 PST,"""c24fb2c11acfc1815a5c7af21e5949…","""2c4e89dc8fd02af69254564d713b17…",2025-11-09 18:21:20.429501 UTC,"""26c3f5ad-cc6b-42ac-a71e-607e26…","""d33fa721-9845-48a5-88b4-334db0…",0
"""0004""","""Barangay4""","""rural""",null,2024-01-01 00:00:00 PST,"""c75aa391ef5a78c314ae965e84acb1…","""2c4e89dc8fd02af69254564d713b17…",2025-11-09 18:21:20.429501 UTC,"""942519ac-724a-415d-9f47-86cbc1…","""3f26b6f7-cd9b-4691-bf9e-76a2aa…",0


In [98]:
with_prev.updated_fields_in_y

psgc_id,psgc_name,settlement_type,status,valid_from,identity_hash,fields_hash,ingestion_datetime,surrogate_id,psgc_id_right,psgc_name_right,settlement_type_right,status_right,valid_from_right,fields_hash_right,ingestion_datetime_right,surrogate_id_right,durable_id,version
str,str,str,str,"datetime[μs, Asia/Manila]",str,str,"datetime[μs, UTC]",str,str,str,str,str,"datetime[μs, Asia/Manila]",str,"datetime[μs, UTC]",str,str,u16
"""0002""","""Barangay2""","""rural""","""poblacion""",2025-01-01 00:00:00 PST,"""7c42c5fb78f5c1861a0276de94f6ae…","""2ed2f2fb3470a0e8c9d1da988e6b30…",2025-11-09 18:21:20.431038 UTC,"""6fe57ef5-15ee-4eb5-997a-8b2a29…","""0002""","""Barangay2""","""urban""","""poblacion""",2024-01-01 00:00:00 PST,"""cb52d0bd57a59bcc88133d10efb5da…",2025-11-09 18:21:20.429501 UTC,"""70b72584-6ddf-4e2e-8cf8-ed2852…","""d62228dd-67a0-4fa1-90a8-63c73c…",0


In [ ]:
barangay.sample(10)

In [ ]:
from pprint import pprint

for name, _ in tables.items():
    print(f"FOR TABLE: {name}")
    pprint(
        {col: dtype for col, dtype in zip(tables[name].columns, tables[name].dtypes)},
        sort_dicts=False,
    )
    print()

# 🥇 Gold

In [ ]:
from pap_datalab.engine import PapDatalab

lab = PapDatalab(environment="dev", environment_path="../dev.env")
client = lab.get_client(database="depdev")

In [ ]:
for table_name, table in tables.items():
    print(f"Writing table to ClickHouse: {table_name}")
    for i in range(0, len(table), 5000):
        chunk = table[i : i + 5000]
        client.insert_arrow(table=table_name, arrow_table=chunk.to_arrow())